In [3]:
import numpy as np
import hdbscan
import pickle
import pandas as pd
from itertools import product

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [5]:
X_umap = np.load("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb10/umap_coords_inlier.npy")

with open("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb10/skenario_d_results.pkl", "rb") as f:
    results_d = pickle.load(f)

print(f"X_umap shape : {X_umap.shape}")
print(f"Keys pkl     : {list(results_d.keys())}")

X_umap shape : (14790, 30)
Keys pkl     : ['labels_D', 'labels_D_pre_knn', 'lof_is_inlier', 'lof_scores', 'umap_coords_inlier', 'idx_inlier', 'idx_outlier', 'params', 'metrics']


In [6]:
MCS_LIST = [5, 10, 15, 25, 50]
MS_LIST  = [2, 5, 10]

rows = []
for mcs, ms in product(MCS_LIST, MS_LIST):
    if ms > mcs:
        continue  # constraint HDBSCAN: ms <= mcs

    clu = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        min_samples=ms,
        metric='euclidean',
        cluster_selection_method='eom',
        gen_min_span_tree=False
    )
    labels = clu.fit_predict(X_umap)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = int((labels == -1).sum())
    coverage   = (labels >= 0).sum() / len(labels)

    if n_clusters >= 2:
        try:
            dbcv = hdbscan.validity_index(X_umap.astype(np.float64), labels)
        except Exception as e:
            dbcv = float('nan')
    else:
        dbcv = float('nan')

    rows.append(dict(mcs=mcs, ms=ms, n_clusters=n_clusters,
                     n_noise=n_noise, coverage_pct=round(coverage*100,1),
                     dbcv=round(dbcv,4)))
    print(f"mcs={mcs:3d} ms={ms:2d} → k={n_clusters:4d} noise={n_noise:4d} cov={coverage*100:.1f}% DBCV={dbcv:.4f}")

mcs=  5 ms= 2 → k= 380 noise=2135 cov=85.6% DBCV=0.6496
mcs=  5 ms= 5 → k= 238 noise=1994 cov=86.5% DBCV=0.7045
mcs= 10 ms= 2 → k= 214 noise=1699 cov=88.5% DBCV=0.6744
mcs= 10 ms= 5 → k= 162 noise= 781 cov=94.7% DBCV=0.7202
mcs= 10 ms=10 → k= 136 noise= 446 cov=97.0% DBCV=0.8356
mcs= 15 ms= 2 → k= 150 noise= 814 cov=94.5% DBCV=0.6446
mcs= 15 ms= 5 → k= 142 noise= 849 cov=94.3% DBCV=0.7095
mcs= 15 ms=10 → k= 123 noise= 450 cov=97.0% DBCV=0.8391
mcs= 25 ms= 2 → k= 104 noise= 559 cov=96.2% DBCV=0.7152
mcs= 25 ms= 5 → k= 105 noise= 518 cov=96.5% DBCV=0.7634
mcs= 25 ms=10 → k=  99 noise= 421 cov=97.2% DBCV=0.8032
mcs= 50 ms= 2 → k=  76 noise= 870 cov=94.1% DBCV=0.7469
mcs= 50 ms= 5 → k=  77 noise= 817 cov=94.5% DBCV=0.7399
mcs= 50 ms=10 → k=  77 noise= 744 cov=95.0% DBCV=0.7508


In [7]:
df = pd.DataFrame(rows).sort_values('dbcv', ascending=False)
print(df.to_string(index=False))

 mcs  ms  n_clusters  n_noise  coverage_pct   dbcv
  15  10         123      450          97.0 0.8391
  10  10         136      446          97.0 0.8356
  25  10          99      421          97.2 0.8032
  25   5         105      518          96.5 0.7634
  50  10          77      744          95.0 0.7508
  50   2          76      870          94.1 0.7469
  50   5          77      817          94.5 0.7399
  10   5         162      781          94.7 0.7202
  25   2         104      559          96.2 0.7152
  15   5         142      849          94.3 0.7095
   5   5         238     1994          86.5 0.7045
  10   2         214     1699          88.5 0.6744
   5   2         380     2135          85.6 0.6496
  15   2         150      814          94.5 0.6446
